# M3 · Tokenizers & chat templates

In M2 the model never saw your text. It saw a list of numbers. This lesson is about how text becomes those numbers, and what else gets slipped in on the way.

That matters more than it sounds. When you fine-tune a chat model later (M5 and M10), your training text has to be turned into numbers in exactly the way the model expects. Get it slightly wrong and nothing crashes: the model just learns worse, and you may never find out why.

By the end of this lesson you'll have:

- seen how text becomes bytes, and bytes become tokens
- built a small tokenizer of your own, the same kind real models use
- looked inside a real tokenizer and its special tokens
- seen exactly what a chat model receives when you send it a conversation

**How to work through it** (same as M1 and M2)

- Run each cell from top to bottom with **Shift-Enter**, and read the **What each line does** box under it.
- When a cell says **Predict**, write your guess down before running it.
- New words are **bold** where they're explained, and collected in the glossary in `README.md`.
- Save with **Cmd-S** now and then so the dashboard sees your progress.

Sections 0 to 3 are plain Python. Section 4 downloads a tokenizer (about 3 MB, no model weights).

## 0. What the model sees

In M2 you called `tokenizer.encode(...)` and got back a list of numbers. Each number is a **token id**: the position of one **token** (a small piece of text) in the model's **vocabulary**, the full list of tokens it knows. The model only ever sees those ids.

Computers already have a number for every character. Python's `ord` gives it to you:

In [ ]:
print([ord(c) for c in "Hi!"])
print([ord(c) for c in "é🙂"])

**What each line does**

- `[ord(c) for c in "Hi!"]`: a list comprehension (M1 section 1). It goes through the text one character at a time and collects `ord(c)`, the character's number.
- `H` is 72, `i` is 105 and `!` is 33. Characters from other languages, and emoji, get much bigger numbers: 🙂 is 128578.

So why not just use one token per character? Two problems:

- There are about 150,000 characters, so the vocabulary would be huge, and most of it would hardly ever be used.
- Every word would take many tokens. The model does work for every token, so long sequences are slow and use more memory.

The opposite, one token per word, has its own problems: there are millions of words, and a new word (a name, a typo, a new product) would have no token at all.

Real tokenizers sit in between. Common words get one token each, and rare words are built from smaller pieces. The method behind most of them is called BPE, and you'll build it in section 2. It starts from bytes, so first, bytes.

## 1. Characters, bytes and UTF-8

A **byte** is a number from 0 to 255. Everything a computer stores is bytes, text included.

**UTF-8** is the standard way to turn text into bytes. English letters take one byte each. Accented letters take two, most other scripts (Hindi, Chinese, Arabic...) take two or three, and emoji take four.

**Predict:** how many bytes is 🙂?

In [ ]:
for text in ["A", "é", "न", "🙂"]:
    data = text.encode("utf-8")
    print(text, list(data), len(data), "bytes")

**What each line does**

- `text.encode("utf-8")`: turns text into bytes using UTF-8. The result is a `bytes` object.
- `list(data)`: shows the bytes as a list of plain numbers, each between 0 and 255.
- `len(data)`: how many bytes. 1 for `A`, 2 for `é`, 3 for `न` and 4 for 🙂.

Going the other way, from bytes back to text, is `.decode("utf-8")`. But only whole characters can be decoded. If you cut a character in half, those bytes don't mean anything on their own:

In [ ]:
smile = "🙂".encode("utf-8")
print(smile.decode("utf-8"))
print(bytes([72, 105]).decode("utf-8"))
print(smile[:2].decode("utf-8", errors="replace"))

**What each line does**

- `smile.decode("utf-8")`: bytes back to text. The four bytes become 🙂 again.
- `bytes([72, 105])`: builds bytes from a list of numbers. 72 and 105 decode to `Hi`.
- `smile[:2]`: only the first two of the emoji's four bytes. Plain `.decode` would raise an error. `errors="replace"` shows the replacement character � instead.
- You'll need `errors="replace"` in exercise 5: a token can end halfway through a character.

Here's the useful part. Any text at all, in any language, turns into bytes, and there are only 256 different bytes. So a tokenizer that starts from bytes can handle any text: nothing is ever unknown. The worst case is one token per byte.

BPE starts there and makes it shorter.

## 2. BPE by hand

**BPE** (byte pair encoding) learns its tokens from example text. The whole method is:

1. Start with the text as bytes: ids 0 to 255.
2. Count every pair of neighbouring ids.
3. Take the most common pair, give it a new id (256 for the first one), and replace that pair everywhere with the new id. That's one **merge**.
4. Repeat, with the next new id each time (257, 258, ...).

The list of merges you learn *is* the tokenizer. Real models do this with tens of thousands of merges on huge amounts of text; SmolLM2 has 49,152 tokens. We'll start with the classic tiny example.

In [ ]:
text = "aaabdaaabac"
ids = list(text.encode("utf-8"))
print(ids)

for pair in zip(ids, ids[1:]):
    print(pair, end=" ")

**What each line does**

- `ids = list(text.encode("utf-8"))`: the text as byte ids. `a` is 97, `b` 98, `c` 99 and `d` 100.
- `ids[1:]`: the same list without its first item. `zip(ids, ids[1:])` walks both lists together, so you get each id next to the one after it: (97, 97), (97, 97), (97, 98), ... These are the neighbouring pairs.
- `print(..., end=" ")`: prints on one line, separated by spaces, instead of a new line each time.

Now count them. A dict works well for that: the pair is the key, and its count is the value.

In [ ]:
counts = {}
for pair in zip(ids, ids[1:]):
    counts[pair] = counts.get(pair, 0) + 1
print(counts)

top = max(counts, key=counts.get)
print("most common:", top, "seen", counts[top], "times")

**What each line does**

- `counts = {}`: an empty dict.
- `counts.get(pair, 0)`: the count so far, or 0 if this pair hasn't been seen yet. Add 1 and store it back.
- The dict keeps pairs in the order they were first seen. That matters for the next line.
- `max(counts, key=counts.get)`: the key with the biggest count. `key=counts.get` tells `max` to compare the counts, not the pairs themselves. If two pairs tie, `max` keeps the first one it saw.
- `(97, 97)`, which is `aa`, is the most common pair: 4 times.

Now the merge: walk along the ids, and wherever that pair appears, write the new id instead of the two ids.

A plain `for` loop can't do this, because after a merge you need to jump ahead by *two*. So this uses a `while` loop with its own position counter `i`:

In [ ]:
new_id = 256
merged, i = [], 0
while i < len(ids):
    if i + 1 < len(ids) and (ids[i], ids[i + 1]) == top:
        merged.append(new_id)
        i += 2
    else:
        merged.append(ids[i])
        i += 1

print(ids)
print(merged)

**What each line does**

- `merged, i = [], 0`: an empty list for the result, and a position counter starting at 0 (two assignments in one line).
- `while i < len(ids):`: keeps going until `i` runs off the end of the list.
- `i + 1 < len(ids)`: makes sure there *is* a next id before looking at `ids[i + 1]`. Without it, the last step would read past the end and crash.
- `(ids[i], ids[i + 1]) == top`: is the pair at this position the one we're merging? If so, write `new_id` and jump past both (`i += 2`).
- Otherwise copy the id unchanged and move on by one (`i += 1`).
- 11 ids became 9. Notice `aaa` became `256, 97`, not `256, 256`: pairs don't overlap.

That was one round. Each further round does the same on the new list: count, pick the top pair, merge it with the next id. After three rounds this text goes:

| Round | Merge | Ids |
|---|---|---|
| start | | `97 97 97 98 100 97 97 97 98 97 99` (11) |
| 1 | `(97, 97)` → 256, which is `aa` | `256 97 98 100 256 97 98 97 99` (9) |
| 2 | `(256, 97)` → 257, which is `aaa` | `257 98 100 257 98 97 99` (7) |
| 3 | `(257, 98)` → 258, which is `aaab` | `258 100 258 97 99` (5) |

In round 2, `(256, 97)` and `(97, 98)` were both seen twice. It's a tie, and `max` kept `(256, 97)` because it was seen first. Real tokenizers break ties in different ways, but each one always uses the same rule, so training twice on the same text gives the same tokenizer.

The merges `{(97, 97): 256, (256, 97): 257, (257, 98): 258}` are the whole tokenizer. In exercises 1 to 3 you'll turn the cells above into functions and repeat the rounds.

## 3. Encoding and decoding

To **encode** is to turn text into token ids; to **decode** is to turn token ids back into text. A tokenizer has to do both, and decoding what you encoded must give back exactly the text you started with.

**Decoding** is the easy one. Every id stands for some bytes: ids 0 to 255 are one byte each, and each merged id is the bytes of its two parts joined together. Build that table, look up each id, join the bytes, and decode them as UTF-8.

In [ ]:
merges = {(97, 97): 256, (256, 97): 257, (257, 98): 258}

vocab = {i: bytes([i]) for i in range(256)}
for (a, b), new_id in merges.items():
    vocab[new_id] = vocab[a] + vocab[b]

print(vocab[256], vocab[257], vocab[258])
tokens = [258, 100, 258, 97, 99]
print(b"".join(vocab[t] for t in tokens).decode("utf-8", errors="replace"))

**What each line does**

- `{i: bytes([i]) for i in range(256)}`: a dict comprehension (like a list comprehension, but it builds a dict). Each id 0 to 255 maps to its single byte.
- `for (a, b), new_id in merges.items():`: goes through the merges in the order they were learned. `(a, b)` unpacks the pair into its two ids.
- `vocab[a] + vocab[b]`: joins two `bytes` objects. Because merges are in learned order, both parts are always in `vocab` already.
- `b"".join(...)`: joins many `bytes` objects into one. The `b` in front makes it an empty `bytes`, not an empty string.
- The five tokens decode back to `aaabdaaabac`.

**Encoding** new text uses the merges you learned, and the order matters. Start from the bytes, and at every step apply the merge that was learned *first* (the lowest new id) among the pairs present. That rebuilds the tokens in the same order training built them.

Here's why "leftmost pair first" would be wrong. Suppose training learned `bc` first (256), then `ab` (257). Encoding `abc`:

In [ ]:
order = {(98, 99): 256, (97, 98): 257}
ids = list("abc".encode("utf-8"))

present = list(zip(ids, ids[1:]))
best = min(present, key=lambda p: order.get(p, float("inf")))
print("pairs present:", present)
print("apply first:", best, "->", order[best])

**What each line does**

- `order`: two merges. `(98, 99)` is `bc`, learned first. `(97, 98)` is `ab`, learned second.
- `present`: the neighbouring pairs in `abc`: `(97, 98)` and `(98, 99)`.
- `lambda p: ...` is a tiny function written in one line: it takes a pair `p` and returns `order.get(p, float("inf"))`, the pair's merge id.
- `float("inf")` is infinity, bigger than every number. A pair that was never learned gets infinity, so it never wins.
- `min(present, key=...)`: the pair with the lowest merge id. It's `(98, 99)`, so the result is `a` + `bc` = `[97, 256]`. Merging the leftmost pair first would give `ab` + `c` instead: different tokens for the same text.

Encoding repeats that step (pick the lowest-id pair, merge it, using your merge loop from section 2) until no pair present is in the merges. If even the best pair has merge id infinity, you're done.

That's everything for exercises 4 and 5.

## 4. A real tokenizer

Now a real one: the tokenizer of SmolLM2-135M-Instruct, the chat version of the model from M2. It's a byte-level BPE like yours, with 49,152 tokens instead of a handful.

The first run downloads its files (about 3 MB, no model weights) and caches them.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M-Instruct")
print(len(tokenizer), "tokens in the vocabulary")

print(tokenizer.tokenize("Hello world, hello!"))
print(tokenizer.encode("Hello world, hello!"))

**What each line does**

- `AutoTokenizer.from_pretrained(...)`: downloads (or reuses) a model's tokenizer, as in M2.
- `len(tokenizer)`: the size of the vocabulary.
- `tokenizer.tokenize(text)`: the tokens as text pieces, so you can see where it cut. `tokenizer.encode(text)` gives their ids.
- `Ġ` is how this tokenizer shows a space: `Ġworld` means ` world`, with the space in front. Most tokens carry the space before a word.
- `Hello` and `Ġhello` are different tokens. Capital letters and spaces both change the token.

**Predict:** how many tokens is each of these? Write your guesses down, then run the cell.

`hello`, ` hello`, `Hello`, `12345`, `unbelievable`, `🙂`, `नमस्ते`

In [ ]:
for text in ["hello", " hello", "Hello", "12345", "unbelievable", "🙂", "नमस्ते"]:
    ids = tokenizer.encode(text)
    pieces = [tokenizer.decode([i]) for i in ids]
    print(f"{text!r:16} {len(ids)} tokens  {ids}  {pieces}")

**What each line does**

- `tokenizer.decode([i])`: decodes one token at a time, so you can see what each one stands for.
- `f"{text!r:16}"`: `!r` shows the text with quotes (so you can see the space in `' hello'`), and `:16` pads it to 16 characters so the columns line up.
- `12345` is five tokens: SmolLM2 splits numbers into single digits. That keeps arithmetic consistent, but long numbers cost many tokens.
- `unbelievable` is three pieces: `un`, `belie`, `vable`. Rarer words are built from smaller pieces.
- 🙂 is two tokens, and each one alone decodes to �: a token can end halfway through a character, just like `smile[:2]` in section 1.
- `नमस्ते` takes six tokens, about one per character, while an English word is usually one.

That last point has a real cost. Text in languages the tokenizer saw less of in training takes more tokens, so it's slower, more expensive, and uses up more of the model's context. Compare the same sentence in English and Hindi:

In [ ]:
for text in ["The bread is ready.", "रोटी तैयार है।"]:
    n_bytes = len(text.encode("utf-8"))
    n_tokens = len(tokenizer.encode(text))
    print(
        f"{text:22} {n_bytes} bytes -> {n_tokens} tokens ({n_bytes / n_tokens:.1f} bytes per token)"
    )

**What each line does**

- `n_bytes / n_tokens`: how many bytes each token covers on average. Higher means the tokenizer compresses that text better.
- English gets about 4 bytes per token; Hindi about 2. The run step reports this number for your own tokenizer too.

## 5. Special tokens

Some tokens aren't text at all. A **special token** is a marker with its own id that the tokenizer never splits up. Models learn what each one means during training:

- **BOS** (beginning of sequence) marks where a text starts.
- **EOS** (end of sequence) marks where it ends. In M2 this was the end-of-text token that stopped generation.
- `<|im_start|>` and `<|im_end|>` mark where each chat message starts and ends (section 6).

In [ ]:
print(tokenizer.all_special_tokens)
print("BOS:", tokenizer.bos_token, " EOS:", tokenizer.eos_token, " PAD:", tokenizer.pad_token)

print(tokenizer.encode("<|im_start|>"))
print(tokenizer.tokenize("<im_start>"))

**What each line does**

- `tokenizer.all_special_tokens`: this tokenizer's special tokens.
- `bos_token`, `eos_token`, `pad_token`: which special token plays each role. In SmolLM2-Instruct, `<|im_start|>` is the BOS and `<|im_end|>` is the EOS.
- `encode("<|im_start|>")` is a single id, 1. The tokenizer looks for special tokens as whole strings *before* it splits anything else.
- `<im_start>`, without the bars, isn't special, so it's split like ordinary text into several pieces.

### Padding and the attention mask

To run several texts at once, as one batch, they must all be the same length. **Padding** fills the shorter ones up with a pad token. Then the **attention mask** marks which positions are real (1) and which are padding (0), so the model ignores the padding.

In [ ]:
batch = tokenizer(["Hi", "Hello there friend"], padding=True)
print(batch["input_ids"])
print(batch["attention_mask"])

**What each line does**

- `tokenizer([...], padding=True)`: calling the tokenizer directly encodes a list of texts together, padding the shorter ones to the length of the longest.
- `batch["input_ids"]`: `Hi` is one token, so it got two pad tokens (id 2) added after it.
- `batch["attention_mask"]`: 1 for real tokens and 0 for padding, in the same positions.

Where the padding goes is the **padding side**. It's on the right by default, which is fine for training. For *generation* it must go on the left.

Remember M2: the model writes the next token after the *last* position of each row. With right padding, the last position of the `Hi` row is a pad token, so the model would continue from padding. With left padding, every row ends with its real text.

In [ ]:
tokenizer.padding_side = "left"
batch = tokenizer(["Hi", "Hello there friend"], padding=True)
print(batch["input_ids"])
print(batch["attention_mask"])
tokenizer.padding_side = "right"

**What each line does**

- `tokenizer.padding_side = "left"`: puts padding before the text instead of after it.
- Now both rows end with a real token, so generation continues straight from the text.
- The last line sets it back to the default, so later cells behave as before.

## 6. Chat templates

A chat model doesn't receive a list of messages. It receives one long string, written in a fixed format it learned in training. The **chat template** is the recipe that turns a list of messages into that string. Each model family has its own.

SmolLM2-Instruct uses a format called **ChatML**: every message is wrapped in `<|im_start|>` and `<|im_end|>`, with its role on the first line.

A conversation is a list of dicts, each with a `role` and its `content`. The roles are `user`, `assistant` (the model) and `system`. A **system message** is the instruction at the start that sets how the model should behave.

In [ ]:
messages = [{"role": "user", "content": "Hi!"}]

text = tokenizer.apply_chat_template(messages, tokenize=False)
print(text)

**What each line does**

- `messages`: a conversation with one user message.
- `tokenizer.apply_chat_template(messages, tokenize=False)`: formats the conversation with the model's template. `tokenize=False` returns the string, so you can read it; without it you get token ids.
- Look at the first message: a system message you never wrote, `You are a helpful AI assistant named SmolLM, trained by Hugging Face`. This template adds it whenever the first message isn't a system message.

**Predict:** what changes if you give your own system message?

In [ ]:
messages = [
    {"role": "system", "content": "Answer in one word."},
    {"role": "user", "content": "Capital of France?"},
]
print(tokenizer.apply_chat_template(messages, tokenize=False))

**What each line does**

- With your own system message first, the default one isn't added. Only your messages appear.

### The generation prompt

The formatted chat ends after the last message. To get a reply, the model also needs to see the start of *its* turn: `<|im_start|>assistant` and a newline. That's the **generation prompt**, and `add_generation_prompt=True` adds it.

In [ ]:
messages = [{"role": "user", "content": "Hi!"}]
without = tokenizer.apply_chat_template(messages, tokenize=False)
with_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(repr(with_prompt[len(without) :]))

**What each line does**

- `without` and `with_prompt`: the same chat, formatted without and with the generation prompt.
- `with_prompt[len(without):]`: everything after the first `len(without)` characters, which is exactly the part that was added.
- `repr(...)`: shows the string with its newline as `\n`, so you can see exactly what's there.
- The added part is `<|im_start|>assistant\n`. Use it when you want the model to reply; leave it off when the conversation is already complete, as in training data.

The template itself is a short program, written in a templating language called Jinja. You don't need to learn Jinja: just notice it's a loop over the messages, with the default system message and the generation prompt as two `if`s.

In [ ]:
print(tokenizer.chat_template)

**What each line does**

- `tokenizer.chat_template`: the template, as text. `{% for message in messages %}` loops over the messages, `{{ ... }}` writes text out, and `{% if ... %}` adds something only when a condition is true.
- Exercise 6 is this template, written in Python.

Finally, what the model really receives: the formatted string, as tokens.

In [ ]:
ids = tokenizer.apply_chat_template(messages, add_generation_prompt=True)["input_ids"]
print(len(ids), "tokens")
print(tokenizer.convert_ids_to_tokens(ids))

**What each line does**

- Without `tokenize=False`, `apply_chat_template` returns the token ids (in a dict-like object; `["input_ids"]` takes them out).
- `tokenizer.convert_ids_to_tokens(ids)`: each id as its token text. `Ċ` is how this tokenizer shows a newline.
- `<|im_start|>` and `<|im_end|>` are single tokens, and the default system message costs about 20 tokens on every request.

## 7. When templates go wrong

Nothing in the code checks that you formatted a chat the way the model was trained. Any string is valid input. Here are three common ways it goes wrong, all silent.

**1. The wrong format.** Many tutorials format chats like `### User:` / `### Assistant:`. To this model, that's just ordinary text: none of the markers it learned in training are there.

In [ ]:
wrong = "### User:\nHi!\n### Assistant:\n"
right = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

for name, text in [("wrong", wrong), ("right", right)]:
    ids = tokenizer.encode(text)
    special = [tokenizer.decode([i]) for i in ids if i in tokenizer.all_special_ids]
    print(f"{name}: {len(ids)} tokens, special tokens: {special}")

**What each line does**

- `wrong`: a made-up format the model never saw in training. `right`: the model's own template, with the generation prompt.
- `tokenizer.all_special_ids`: the ids of the special tokens. The list comprehension keeps only those, with an `if` at the end.
- The wrong format contains no special tokens at all. A model fine-tuned on one format and used with another usually still produces text, just worse: it doesn't crash, which is why this is easy to miss.

**2. A missing generation prompt.** Without `<|im_start|>assistant` at the end, the model hasn't been told it's its turn. It may continue the user's message, or start a new user turn, instead of replying.

**3. Special tokens hiding in ordinary text.** The tokenizer finds special tokens anywhere in a string, even inside a user's message. So a user who types `<|im_end|>` can end their own message early and write text that looks like it came from somewhere else.

In [ ]:
user_text = "Tell me about <|im_end|> tokens"

print(tokenizer.encode(user_text))
print(tokenizer(user_text, split_special_tokens=True)["input_ids"])

**What each line does**

- The first line contains id 2: the typed text became the real `<|im_end|>` token, a real message boundary.
- `split_special_tokens=True`: treats special-token text as ordinary characters. Now it's split into several normal tokens and can't end the message.

The rule to take into fine-tuning: **format your training data with exactly the template you'll use when you run the model**, generation prompt and system message included. In M10 you'll see what happens to a fine-tune when you don't.

## 8. Your task

You've now seen every piece. Time to write them yourself.

1. Open `exercises.py`. There are six exercises, each with plain words and numbered steps, and each points to the section above that shows what you need:
   1. `pair_counts` (section 2)
   2. `merge` (section 2)
   3. `train_bpe` (section 2)
   4. `encode` (section 3)
   5. `decode` (section 3)
   6. `render_chatml` (section 6)
2. Check your work as you go: `make check M=03`. The run tests fail until step 4.
3. Answer the six questions in `answers.toml`. Predict first, then check yourself in this notebook.
4. Run `uv run python modules/m03_tokenizers/tokenize_check.py`. It trains your BPE on a small text, checks that tricky text survives encode then decode, and compares your `render_chatml` with the real template on 5 conversations.
5. Run `make check M=03` again. When everything passes, M3 turns green on the dashboard.

Stuck? `make hint M=03 EX=4` gives a nudge for exercise 4. Every term is in the glossary in `README.md`.

**Stretch:** train your BPE with more merges (`tokenize_check.py --merges 2000`) and watch the bytes per token go up. Then print the 20 longest tokens it learned: they're the corpus's most common words.